# Language model LSTM tiếng Việt: byte, BPE và bits-per-character

Bạn train language model LSTM trên Wikipedia tiếng Việt với nhiều cách tokenize (byte, BPE vocab nhỏ, BPE vocab lớn),
rồi trả lời câu hỏi mà mọi bảng xếp hạng LLM phải trả lời: **so perplexity giữa hai tokenizer thế nào cho công bằng?**
Module 05 train GPT mini trên đúng dữ liệu này; con số bits-per-character của bạn là baseline để nó vượt qua.

**Cách làm bài**
- Bốn hàm được test chấm (cell dưới, đặc tả trong docstring). Phần còn lại tự thiết kế.
- GPU: chạy bằng kernel `ai-gpu` (xem `setup/gpu/README.md`); không có GPU thì thu nhỏ (xem README của bài).
- Chấm: `uv run pytest tracks/ai-engineer/modules/04-nlp-sequence-models/exercises/advanced/02-lm-perplexity -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, tải Wikipedia tiếng Việt nếu chưa có
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

WIKI = ensure("wikipedia-vi")

In [ ]:
import math

import torch

## 1. Bốn hàm được chấm

- `batchify(ids, batch_size) -> LongTensor (batch_size, n)`: cắt bỏ phần dư cuối, dòng `b` là đoạn liên tục thứ `b`
  của chuỗi (`ids[b*n:(b+1)*n]`), để truncated BPTT chuyền được hidden state từ đoạn này sang đoạn sau của cùng dòng.
- `get_batch(data, i, bptt) -> (x, y)`: `x = data[:, i:i+L]`, `y` là `x` dịch phải một vị trí, với
  `L = min(bptt, n - 1 - i)`.
- `nll_to_metrics(total_nll, n_tokens, n_chars) -> dict`: từ tổng negative log-likelihood (nat) trên tập đánh giá,
  trả `{"ppl": ..., "bits_per_token": ..., "bpc": ...}`: perplexity theo token, bit mỗi token, bit mỗi **ký tự** văn bản.
- `sample_next(logits, temperature, top_k, generator) -> int`: `temperature == 0` thì argmax; ngược lại chia logits cho
  temperature, nếu `top_k` không phải `None` thì chỉ giữ `top_k` logits lớn nhất (giá trị bằng ngưỡng vẫn giữ), rồi
  `torch.multinomial` với `generator`.

In [ ]:
def batchify(ids: list[int], batch_size: int) -> torch.Tensor:
    # TODO
    raise NotImplementedError


def get_batch(data: torch.Tensor, i: int, bptt: int) -> tuple[torch.Tensor, torch.Tensor]:
    # TODO
    raise NotImplementedError


def nll_to_metrics(total_nll: float, n_tokens: int, n_chars: int) -> dict[str, float]:
    # TODO
    raise NotImplementedError


def sample_next(logits: torch.Tensor, temperature: float, top_k: int | None, generator: torch.Generator) -> int:
    # TODO
    raise NotImplementedError

## 2. Gợi ý bố cục (không chấm)

1. Dữ liệu: 1 000 bài dài (≥ 2 000 ký tự) lấy ngẫu nhiên từ shard `train-00002-of-00004` làm train, 100 bài khác làm
   validation; NFC; nối bài bằng `"\n\n"`.
2. Tokenizer: byte (256), BPE byte-level của `tokenizers` vocab 2 000 và 8 000 (train trên tập train). So số ký tự mỗi token.
3. Model: LSTM 2 tầng 512, embedding 256, dropout 0.2; Adam 2e-3 cosine, clip 0.5; truncated BPTT 128, batch 32;
   **cùng số bước** cho mọi tokenizer.
4. Đánh giá trên validation: tổng NLL → `nll_to_metrics`. Bảng: perplexity theo token và bpc.
5. Sinh văn bản từ "Hà Nội là" với vài temperature, top-k. Viết `REPORT.md`.

In [ ]:
# explore: TODO: pipeline của bạn